# 03 · CANN: nesting the GLM in a network
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/03_cann.ipynb)

$\lambda^{\mathrm{CANN}}(x)=\lambda^{\mathrm{GLM}}(x)\cdot\exp\{w_0+\langle w,z^{(d:1)}(x)\rangle\}$, initialised at $w=0$: training starts exactly at the GLM.

In [ ]:
# Setup: works in Colab, the dev container and a local clone.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("/content/deep-actuarial-science"):
    !git clone -q https://github.com/actuarialsociety/deep-actuarial-science /content/deep-actuarial-science
    !pip install -q -e /content/deep-actuarial-science
if "google.colab" in sys.modules:
    os.chdir("/content/deep-actuarial-science")
    sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
run("frequency", models=["glm", "cann"])
leaderboard("frequency", ["glm", "cann"])

In [ ]:
load("cann.freq_reconciliation")

## Open the box: zero initialisation reproduces the GLM
Build a CANN with 0 epochs: its predictions equal the GLM's to float precision.

In [ ]:
import numpy as np
from deep_actuarial_science.models.frequency import CANN
glm, test = load("glm.freq_model"), load("freq_test")
cann0 = CANN().fit(load("freq_learn"), {"epochs": 0}, base=glm)
np.abs(cann0.predict(test) / glm.predict(test) - 1).max()

## Where does the network correct the GLM?
The ratio CANN / GLM by covariate shows what the GLM misspecifies (e.g. the young-driver and bonus-malus shapes).

In [ ]:
p = load("cann.freq_predictions").assign(glm=load("glm.freq_predictions").pred)
p = p[p.split == "test"]
r = p.groupby(p.DrivAge.clip(upper=90))[["pred", "glm"]].sum()
(r.pred / r.glm).plot(title="CANN / GLM by driver age", ylabel="multiplicative correction");

In [ ]:
show("frequency", "cann", ["calibration", "marginal_drivage", "training"])

## Exercise
Freeze less: make the GLM part trainable (hint: subclass `CANN` and make the offset a learnable linear layer). Does joint training beat the offset version?

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. In Colab the UI opens below and in a new tab through Colab's proxy (MLflow runs on the Colab machine, so a plain localhost link would point at your own computer).

In [ ]:
mlflow_ui()